# 🧠 LLM Fine-Tuning & Quantization — Complete Notes

---

## 🌍 1. Introduction: Why We Fine-Tune & Quantize

Large Language Models (LLMs) like **LLaMA**, **Mistral**, or **GPT** have billions of parameters, making them powerful but very heavy to train or deploy.  
To make them usable on smaller systems or specialized for specific tasks, we apply:

- **Fine-Tuning** → teaches *new behavior or domain knowledge*  
- **Quantization** → *compresses* the model to reduce memory and increase speed

---

## ⚙️ 2. Fine-Tuning — Teaching a Pretrained Model

### 🧩 Concept
Fine-tuning starts with a **pretrained model** that already understands general language.  
We train it further on **domain-specific data** (like law, medicine, finance, etc.) to specialize it.

### 🔁 Process
1. Load pretrained model (e.g., LLaMA-2)
2. Provide dataset → prompts and expected responses
3. Train → model learns corrections via backpropagation
4. Save fine-tuned model → behaves in domain-specific way

### 🎯 Goal
Change **what** the model knows (behavior & domain understanding).

---

## ⚙️ 3. Quantization — Compressing the Model

### 🧩 Concept
Quantization converts model weights from **higher-precision numbers** (e.g., 32-bit floats)  
to **lower-precision integers** (e.g., 8-bit or 4-bit), reducing memory and speeding inference.

| Type | Bits | Example Use |
|-------|------|-------------|
| FP32 | 32-bit float | Training, high precision |
| FP16 | 16-bit float | Mixed precision training |
| INT8 | 8-bit integer | Inference |
| INT4 | 4-bit integer | Ultra-efficient deployment |

### 📦 Result
- Smaller model size  
- Faster inference  
- Slight loss of precision (usually negligible)

### 🎯 Goal
Change **how** the model’s knowledge is *stored and computed*, not what it knows.

---

## 🧠 4. LoRA — Low-Rank Adaptation for Efficient Fine-Tuning

### 🧩 What It Does
LoRA freezes the pretrained model weights and adds small trainable layers (adapters)  
to learn lightweight updates.

Mathematically:
\[
W' = W + B \times A
\]
where  
- \(W\) = original pretrained weights (frozen)  
- \(A, B\) = small LoRA matrices (trainable)  
- \(r\) = **rank**, controlling adapter capacity

### 💡 Why It’s Powerful
- Fine-tunes large models with tiny memory footprint  
- Stores new knowledge in adapters instead of full model  
- Adapters can be merged, swapped, or stacked for multiple domains

### 🧠 LoRA Training Flow
1. Load base model in FP16 precision  
2. Freeze weights \(W\)  
3. Insert LoRA adapters (A, B)  
4. Train → update only A, B  
5. Merge → \(W' = W + B A\)  
6. Quantize later for deployment (optional)

---

## ⚙️ 5. QLoRA — Quantized LoRA

### 🧩 What It Adds
QLoRA = **LoRA + Quantization at training time**  
It loads the pretrained model in **4-bit (NF4)** format to save GPU memory *before fine-tuning*.

### ⚙️ Workflow
1. Load pretrained model → quantize to 4-bit NF4  
2. Freeze quantized weights  
3. Attach LoRA adapters (A, B) in FP16  
4. Train → update only adapters  
5. Merge or deploy directly

### 💡 Purpose
- Train large models on small GPUs (e.g., 13B–70B on a 24 GB GPU)  
- Achieve nearly identical accuracy to full-precision LoRA

### 📊 LoRA vs QLoRA

| Feature | LoRA | QLoRA |
|----------|------|-------|
| Base model precision | FP16 / FP32 | 4-bit (NF4) |
| Quantization timing | After training | Before training |
| Purpose | Efficient fine-tuning | Memory-efficient fine-tuning |
| VRAM usage | Moderate | Very low |
| Accuracy | Excellent | Almost identical |

---

## ⚙️ 6. PTQ — Post-Training Quantization

### 🧩 What It Is
After you finish fine-tuning, you compress the model (FP16 → INT8/INT4)  
for smaller size and faster inference.

### ⚙️ When To Use
- After LoRA/QLoRA fine-tuning  
- For deployment on low-VRAM GPUs or CPUs

### ⚡ Goal
**Inference optimization** — make the model lighter *after* training.

---

## ⚙️ 7. QAT — Quantization-Aware Training

### 🧩 What It Is
Train the model **while simulating quantization** in the forward pass.  
The model “learns” to tolerate rounding errors and maintains high accuracy after real quantization.

### 🔁 Process
1. Take pretrained or fine-tuned model  
2. Add *fake quantization* layers during training  
3. Train normally (gradients still in FP16)  
4. Export fully quantized model (INT8/INT4)

### 📊 Comparison

| Method | When Quantized | Accuracy | Purpose |
|---------|----------------|-----------|----------|
| **PTQ** | After training | Slight drop | Quick deployment |
| **QAT** | During training | Very high | Train robust quantized models |
| **QLoRA** | During fine-tuning | Very high | Low-VRAM fine-tuning |

---

## ⚙️ 8. Rank (r) in LoRA / QLoRA

### 🧩 Definition
Rank \(r\) = size of the inner dimension in LoRA adapters  
(\(A ∈ ℝ^{r×k}, B ∈ ℝ^{d×r}\)).  
It controls the adapter’s **learning capacity**.

### ⚖️ Trade-off

| Rank | Pros | Cons |
|-------|------|------|
| Low (4–8) | Lightweight, small memory | May underfit complex domains |
| Medium (8–16) | Balanced performance | Standard default |
| High (32+) | Captures complex patterns | Higher VRAM & slower training |

### 💡 Choose rank based on domain complexity & available VRAM.

---

## 🧩 9. Complete Workflow Summary

        ┌──────────────────────────────────────────┐
        │  Pretrained Base Model                   │
        └──────────────────────────────────────────┘
                         │
        ┌────────────────┴────────────────┐
        │                                 │
    ┌──────────┐                     ┌──────────┐
    │   LoRA   │                     │   QLoRA  │
    └──────────┘                     └──────────┘
    - Base in FP16                   - Base quantized to 4-bit
    - Add adapters                   - Add adapters (FP16)
    - Fine-tune adapters             - Fine-tune adapters
                         │
                         ▼
        ┌──────────────────────────────────────────┐
        │  Fine-Tuned Model                        │
        └──────────────────────────────────────────┘
                         │
               ┌───────────────────┐
               │   Post-Training    │
               │   Quantization     │
               └───────────────────┘
                         │
                         ▼
        ┌──────────────────────────────────────────┐
        │  Quantized Model for Deployment (INT8/4) │
        └──────────────────────────────────────────┘


---

## ⚡ 10. Quick Recap

| Concept | Stage | Purpose |
|----------|--------|----------|
| **Fine-Tuning** | After pretraining | Teach new domain/task |
| **LoRA** | During fine-tuning | Efficient training with adapters |
| **QLoRA** | During fine-tuning | Memory-efficient LoRA (4-bit base) |
| **QAT** | During training | Train model to be quantization-robust |
| **PTQ** | After fine-tuning | Compress for inference |
| **Rank (r)** | LoRA/QLoRA hyperparameter | Controls adapter capacity |

---

### 🧠 In One Sentence
> **LoRA** fine-tunes a model efficiently,  
> **QLoRA** fine-tunes it efficiently *and* memory-efficiently,  
> **QAT** trains it to survive quantization,  
> **PTQ** compresses it post-training,  
> and **Rank** controls how much the adapters can learn.